# motivosreprovacao ingestion and bronze loading

In [ ]:
import os
import json
import re
import shutil
import unicodedata
import pandas as pd

from pyspark.sql.functions import (
    col,
    lit,
    current_date,
    current_timestamp,
    regexp_extract,
    regexp_replace,
    to_date
)
from pyspark.sql.utils import AnalysisException



# PARAMETERS

# Defaults. Quando corre pelo pipeline, estes valores são substituídos.
load_mode = "incremental"          # full ou incremental
run_id = "manual"



# CHECK DOS PARÂMETROS RECEBIDOS

print("Parâmetros recebidos pelo notebook:")
print("load_mode:", load_mode)
print("run_id:", run_id)

if load_mode not in ["full", "incremental"]:
    raise ValueError(f"load_mode inválido: {load_mode}")



# FUNÇÕES AUXILIARES

def clean_column_name(col_name):
    c = str(col_name).strip().lower()
    c = unicodedata.normalize("NFD", c).encode("ascii", "ignore").decode("utf-8")
    c = re.sub(r"[\s\-]+", "_", c)
    c = re.sub(r"[^a-z0-9_]", "", c)
    c = re.sub(r"_+", "_", c).strip("_")
    return c



# 1. DEFINIÇÃO DE CAMINHOS

folder_name = "Motivos de Reprovação"

local_input_path = f"/lakehouse/default/Files/{folder_name}"

spark_staging_path = f"Files/{folder_name}/CSV_Staging_Latest"
local_staging_path = f"/lakehouse/default/{spark_staging_path}"
log_file_path = f"{local_staging_path}/_processed_files_log.json"

staging_input_path = f"Files/{folder_name}/CSV_Staging_Latest/*/*.csv"

target_lakehouse = "lh_dsp10"
schema_name = "brz"
table_name = f"{target_lakehouse}.{schema_name}.motivosreprovacao"



# 2. PREPARAR STAGING E LOG

if load_mode == "full":
    print("Modo FULL: limpar staging antigo e ignorar log.")

    if os.path.exists(local_staging_path):
        shutil.rmtree(local_staging_path)

    os.makedirs(local_staging_path, exist_ok=True)

    processed_log = []

elif load_mode == "incremental":
    print("Modo INCREMENTAL: manter staging e ler log.")

    os.makedirs(local_staging_path, exist_ok=True)

    processed_log = []

    if os.path.exists(log_file_path):
        try:
            with open(log_file_path, "r") as f:
                processed_log = json.load(f)
        except Exception as e:
            print(f"Aviso: não foi possível ler o log. Erro: {e}")
            processed_log = []


new_processed_log = list(processed_log)
found_new_data = False



# 3. IDENTIFICAR FICHEIROS EXCEL

excel_files = [
    f for f in os.listdir(local_input_path)
    if f.lower().endswith(".xlsx")
    and os.path.isfile(os.path.join(local_input_path, f))
]

print(f"Ficheiros Excel encontrados: {len(excel_files)}")



# 4. PROCESSAR EXCELS PARA STAGING CSV


target_tab = "Listagem"

for file_name in excel_files:

    if file_name in processed_log:
        print(f"Ficheiro já processado, ignorado: {file_name}")
        continue

    file_path = os.path.join(local_input_path, file_name)
    xl = pd.ExcelFile(file_path)

    if target_tab not in xl.sheet_names:
        print(f"Aba '{target_tab}' não encontrada em {file_name}. Ficheiro ignorado.")
        continue

    print(f"A ler snapshot: {file_name} -> Aba {target_tab}")

    pdf = pd.read_excel(file_path, sheet_name=target_tab)

    if pdf.empty:
        new_processed_log.append(file_name)
        continue

    pdf.columns = [str(c).strip() for c in pdf.columns]

    spark_df = spark.createDataFrame(pdf.astype(str))

    first_col = spark_df.columns[0]

    spark_df = spark_df.filter(
        (col(first_col).isNotNull()) &
        (col(first_col) != "nan") &
        (col(first_col) != "")
    )

    spark_df = (
        spark_df
        .withColumn("meta_source_file", lit(file_name))
        .withColumn("meta_source_sheet", lit(target_tab))
        .withColumn("meta_ingestion_date", current_date().cast("string"))
        .withColumn("audit_run_id", lit(run_id))
        .withColumn("audit_load_mode", lit(load_mode))
    )

    output_dir = f"{spark_staging_path}/{file_name.replace('.xlsx', '')}"

    spark_df.coalesce(1).write \
        .mode("overwrite") \
        .option("header", "true") \
        .csv(output_dir)

    new_processed_log.append(file_name)
    found_new_data = True



# 5. ATUALIZAR LOG

if found_new_data:
    with open(log_file_path, "w") as f:
        json.dump(new_processed_log, f)

    print("Novos snapshots de Motivos de Reprovação processados para staging.")
else:
    print("Não foram encontrados ficheiros novos.")


# 6. LER STAGING E PREPARAR BRONZE

print("Passo 1: A ler dados do staging...")

try:
    df_staging = spark.read.option("header", "true").csv(staging_input_path)

except AnalysisException:
    raise ValueError("Não existem ficheiros CSV no staging para processar.")


print("Passo 2: A limpar nomes de colunas...")

for old_col_name in df_staging.columns:
    new_col_name = clean_column_name(old_col_name)

    if old_col_name != new_col_name:
        df_staging = df_staging.withColumnRenamed(old_col_name, new_col_name)


print("Passo 3: A extrair data do nome do ficheiro...")

df_bronze = (
    df_staging
    .withColumn(
        "meta_source_file_date",
        to_date(
            regexp_replace(
                regexp_extract(
                    col("meta_source_file"),
                    r"(\d{4}_\d{2}_\d{2})",
                    1
                ),
                "_",
                "-"
            )
        )
    )
    .withColumn("audit_brz_load_timestamp", current_timestamp())
    .withColumn("audit_run_id", lit(run_id))
    .withColumn("audit_load_mode", lit(load_mode))
)

df_bronze = df_bronze.filter(
    col("meta_source_file_date").isNotNull()
)





# 8. DEFINIR DADOS A ESCREVER

if load_mode == "full":
    print("Modo FULL: todos os dados do staging serão escritos com overwrite.")
    df_bronze_new = df_bronze

elif load_mode == "incremental":
    print("Modo INCREMENTAL: aplicar left_anti contra Bronze existente.")

    try:
        df_existing = spark.table(table_name).select(
            "meta_source_file"
        ).distinct()

        df_bronze_new = df_bronze.join(
            df_existing,
            on=["meta_source_file"],
            how="left_anti"
        )

    except AnalysisException:
        print("Tabela Bronze ainda não existe. Todos os dados serão inseridos.")
        df_bronze_new = df_bronze



# 9. ESCREVER BRONZE

n_rows = df_bronze_new.count()

if n_rows > 0:

    if load_mode == "full":
        df_bronze_new.write \
            .format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(table_name)

        print(f"FULL load concluído. Tabela {table_name} reconstruída com {n_rows} linhas.")

    elif load_mode == "incremental":
        df_bronze_new.write \
            .format("delta") \
            .mode("append") \
            .option("mergeSchema", "true") \
            .saveAsTable(table_name)

        print(f"INCREMENTAL load concluído. Appended {n_rows} new rows.")

else:
    print("No new data to write to Bronze.")

#### Validation

In [28]:
# from pyspark.sql import functions as F

# df = spark.read.table("lh_dsp10.brz.motivosreprovacao")
# print("Total de linhas em Bronze:", df.count())
# print("Snapshots processados:")
# df.select("meta_source_file_date", "meta_source_file").distinct().orderBy(F.desc("meta_source_file_date")).show(truncate=False)

# # Preview dos dados limpos
# display(df.limit(5))

StatementMeta(, 70fc68c3-45a7-4abf-bab4-48f5ab032cf0, 36, Finished, Available, Finished, False)

Total de linhas em Bronze: 406916
Snapshots processados:
+---------------------+------------------------------------+
|meta_source_file_date|meta_source_file                    |
+---------------------+------------------------------------+
|2025-01-10           |Causas de reprovação 2025_01_10.xlsx|
+---------------------+------------------------------------+



SynapseWidget(Synapse.DataFrame, bd4348ef-8f56-4eab-b73a-99a66ca5df47)

In [30]:
# from pyspark.sql import functions as F

# # 1. Load the Table
# table_name = "lh_dsp10.brz.motivosreprovacao"
# df = spark.read.table(table_name)

# print(f"{'='*60}")
# print(f"EXPLORATION & VALIDATION REPORT: {table_name}")
# print(f"{'='*60}\n")

# # --- Step 1: Structural Analysis ---
# print("1. TABLE STRUCTURE & VOLUME")
# print(f"Total Record Count: {df.count()}")
# df.printSchema()

# # --- Step 2: Data Quality Check (Nulls, 'nan', and Empty Strings) ---

# print("--- CORRECTED Data Quality Check (Exact matches only) ---")
# # We change .contains() to simple equality == 
# quality_metrics = df.select([
#     F.sum(F.when(
#         F.col(c).isNull() | 
#         (F.col(c) == "") | 
#         (F.col(c) == "nan") | 
#         (F.col(c) == "None") |
#         (F.col(c) == "NaN"), 1
#     ).otherwise(0)).alias(c) 
#     for c in df.columns
# ])
# quality_metrics.show() #codigoexploracaoorigem (33,783) is the only column with missing data.


# # --- Step 3: Source File Attribution ---
# print("\n3. SOURCE ANALYSIS (Records per File)")
# df.groupBy("meta_source_file", "meta_source_sheet") \
#     .agg(
#         F.count("*").alias("row_count"),
#         F.min("meta_ingestion_date").alias("ingested_on")
#     ) \
#     .orderBy(F.desc("row_count")) \
#     .show(truncate=False)

# # --- Step 4: Business Logic Discovery (Sample Distribution) ---
# print("\n4. DATA DISTRIBUTION (First 3 Business Columns)")
# # We take the first 3 columns (excluding audit columns) to see the most frequent values
# business_cols = [c for c in df.columns if not c.startswith("meta_") and not c.startswith("audit_")][:3]
# for b_col in business_cols:
#     print(f"\nTop 10 values for column: [{b_col}]")
#     df.groupBy(b_col).count().orderBy(F.desc("count")).show(10)

# # --- Step 5: Visual Data Sample ---
# print("\n5. FINAL DATA PREVIEW (Top 10 rows)")
# display(df.limit(10))

# print(f"\n{'='*60}")
# print("VALIDATION COMPLETE")
# print(f"{'='*60}")

StatementMeta(, 70fc68c3-45a7-4abf-bab4-48f5ab032cf0, 38, Finished, Available, Finished, False)

EXPLORATION & VALIDATION REPORT: lh_dsp10.brz.motivosreprovacao

1. TABLE STRUCTURE & VOLUME
Total Record Count: 406916
root
 |-- departamento: string (nullable = true)
 |-- nome: string (nullable = true)
 |-- nome2: string (nullable = true)
 |-- ncv: string (nullable = true)
 |-- id: string (nullable = true)
 |-- datacontrolo: string (nullable = true)
 |-- numanimais: string (nullable = true)
 |-- especie: string (nullable = true)
 |-- motivorejeicao: string (nullable = true)
 |-- codigoexploracaoorigem: string (nullable = true)
 |-- isdeleted: string (nullable = true)
 |-- ano: string (nullable = true)
 |-- mes: string (nullable = true)
 |-- semana: string (nullable = true)
 |-- column1: string (nullable = true)
 |-- meta_source_file: string (nullable = true)
 |-- meta_source_sheet: string (nullable = true)
 |-- meta_ingestion_date: string (nullable = true)
 |-- meta_source_file_date: date (nullable = true)
 |-- audit_brz_load_timestamp: timestamp (nullable = true)

--- CORRECTED Dat

SynapseWidget(Synapse.DataFrame, a414bd5d-fa41-4b52-9a0b-841c6552a285)


VALIDATION COMPLETE
